# Patient Data Analysis for Synthetic Data Generation

## Purpose

This notebook analyzes a real patient healthcare dataset to extract the statistical characteristics needed to generate a realistic synthetic replica. We analyze both **valid data patterns** (for realistic value generation) and **error patterns** (for realistic error injection).

## Dataset Schema

| Field | Description |
|-------|-------------|
| **EGKVERSICHERTENNUMMER (KVNR)** | German health insurance number (1 letter + 9 digits with checksum) |
| **VORNAME** | First name |
| **NACHNAME** | Last name |
| **Geburtsdatum** | Date of birth (YYYY-MM-DD) |
| **PLZ** | German postal code (5 digits) |

## Analysis Flow

1. **Data Loading** — Load source dataset from SQL Server (filtered to 2020+)
2. **General Dataset Statistics** — Size, completeness, and field-level overview
3. **Privacy-Preserving Distributions** — k-anonymized (k≥100) value distributions for realistic generation
4. **Data Quality Analysis** — Error detection, classification, and distribution patterns
5. **Business Rule Analysis** — Duplicate patterns and integrity violations
6. **Synthesis Parameters** — Compiled parameters for synthetic data generation

## Data Recency Filter

Only patients with records from **Q1 2020 onwards** are analyzed:
- Data quality has improved over time
- Older data patterns are less representative for model evaluation
- Recent data reflects current error rates and value distributions

The filter is applied by joining `PATIENTEN_LOOKUP` with both mapping tables (Abr 1 and Abr 2) to include all billing types.

## Privacy Approach

All extracted statistics are **privacy-preserving**:
- k-anonymity with k≥100 (only values appearing 100+ times)
- Aggregation only (no individual records)
- Binning for sensitive values (dates, geographic data)
- Counts rounded to nearest 100 to prevent fingerprinting

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.column import Column
from pyspark.sql.types import IntegerType, StringType
from typing import Optional
from datetime import datetime

# Privacy constants
K_ANONYMITY = 100  # Minimum frequency threshold for k-anonymity
ROUND_TO = 100     # Round counts to nearest N for privacy

# Data recency filter: Only consider data from 2020 onwards
# Format: YYYYQ (e.g., 20201 = Q1 2020, 20244 = Q4 2024)
# Rationale: Data quality improved over time, older data is less relevant for model evaluation
MIN_QUARTER = 20201  # Q1 2020

def round_count(n: int) -> int:
    """Round count to nearest ROUND_TO for privacy."""
    return round(n / ROUND_TO) * ROUND_TO

## 1. Data Loading

Load the patient dataset from SQL Server. We also define validation functions here (used later in Section 4).

In [ ]:
def validate_kvnr_logic(kvnr: Optional[str]) -> Optional[str]:
    """
    Validates German KVNR (Krankenversichertennummer) using official algorithm.
    
    This function identifies KVNR errors that will be replicated in synthetic data:
    - Missing/null values
    - Format violations (length, character types)
    - Checksum mismatches
    
    Algorithm:
    1. Format: 1 uppercase letter + 9 digits
    2. Convert letter to numeric code (A=01, B=02, ..., Z=26)
    3. Apply Modulo-10 weighted checksum (weights: 1,2,1,2,1,2,1,2,1,2)
    4. Sum of digit products modulo 10 must match check digit
    
    Args:
        kvnr: KVNR string to validate
        
    Returns:
        Error message string if invalid, None if valid
    """
    if kvnr is None or not isinstance(kvnr, str):
        return "Missing or non-string input"
    if len(kvnr) != 10:
        return f"Invalid length: {len(kvnr)} (expected 10)"
    
    letter = kvnr[0]
    serial_digits = kvnr[1:9]
    check_digit_provided = kvnr[9]

    if not letter.isalpha() or not letter.isupper():
        return "First character must be an uppercase letter"
    
    if not (serial_digits.isdigit() and check_digit_provided.isdigit()):
        return "Characters 2-10 must be digits"

    try:
        char_code = ord(letter) - 64 
        if char_code < 1 or char_code > 26:
            return "Invalid first letter"
        
        validation_string = f"{char_code:02d}{serial_digits}"
        weights = [1, 2] * 5 
        total_sum = 0
        
        for i, char_digit in enumerate(validation_string):
            digit = int(char_digit)
            weight = weights[i]
            product = digit * weight
            product_sum = sum(int(d) for d in str(product))
            total_sum += product_sum
            
        calculated_check_digit = total_sum % 10
        
        if int(check_digit_provided) != calculated_check_digit:
            return f"Checksum mismatch (Expected {calculated_check_digit}, Got {check_digit_provided})"
        return None
            
    except Exception as e:
        return f"Validation error: {str(e)}"

# Register as Spark UDF for distributed processing
validate_kvnr_udf = F.udf(validate_kvnr_logic, StringType())

def get_name_check_expr(col_name: str) -> Column:
    """Returns a Spark Column expression for name validation."""
    name_col = F.col(col_name)
    return (
        F.when(name_col.isNull() | (name_col == ""), "Missing name")
        # Regex: Allow A-Z, accents, spaces, hyphens. INVERTED (~) to find errors.
        .when(~name_col.rlike(r"^[A-Za-z\u00C0-\u017F\s\-]+$"), "Name contains invalid characters")
        .otherwise(None)
    )

def get_plz_check_expr(col_name: str) -> Column:
    """Returns a Spark Column expression for PLZ validation."""
    plz_col = F.col(col_name)
    plz_str = F.trim(plz_col.cast(StringType()))
    plz_int = plz_col.cast(IntegerType())
    
    return (
        F.when(plz_col.isNull() | (plz_str == ""), "Missing PLZ")
        .when(
            ~plz_str.rlike(r"^\d{5}$"),
            F.concat(F.lit("Invalid format: "), plz_str, F.lit(" (expected 5 digits)")),
        )
        # Check range 1000 to 99998
        .when((plz_int < 1000) | (plz_int > 99998), "PLZ out of valid numeric range")
        .otherwise(None)
    )

def get_date_check_expr(col_name: str) -> Column:
    """Returns a Spark Column expression for Date validation."""
    # Attempt to cast to date; if format is wrong, this becomes NULL
    date_col = F.col(col_name)
    d_col = F.to_date(date_col, "yyyy-MM-dd")
    current_d = F.current_date()
    
    return (
        F.when(date_col.isNull(), "Missing date")
        .when(d_col.isNull(), "Invalid format (expected YYYY-MM-DD)")
        .when(d_col > current_d, "Date is in the future")
        .when(F.year(d_col) < (F.year(current_d) - 120), "Date is > 120 years ago")
        .otherwise(None)
    )

**Source**: `M20_HIS_T_01AB000_PATIENTEN_LOOKUP` table (filtered via mapping tables)

**Note**: We query from PATIENTEN_LOOKUP joined with both mapping tables (Abr 1 + Abr 2) to filter by `DWH_ZEITRAUM`. This includes patient variants from all billing types that have records from 2020 onwards.

In [ ]:
# Create Spark session
spark = (
    SparkSession.builder
        .appName("PatientDataValidation")
        .config("spark.cores.max", 16)
        .config("spark.driver.memory", "2g")
        .config("spark.executor.memory", "16g")
        .config("spark.shuffle.readHostLocalDisk", "false")
        .getOrCreate()
)

# Database connection parameters
server = "do-dwh-db1.doms.kvwl.de"
database = "DWH_HIS1P"
schema = "dbo"
table = "M20_HIS_T_01AB000_PATIENTEN_LOOKUP"  # Patient variants from all billing types

# Mapping tables for time-based filtering (contain DWH_ZEITRAUM)
mapping_table_abr1 = "M20_HIS_T_01AB100_PATID_MAPPING"
mapping_table_abr2 = "M20_HIS_T_01AB200_PATID_MAPPING"

# Database credentials
username = "spark"  # Replace with your username
password = "xxx"  # Replace with your password

# JDBC connection URL (READ ONLY for safety)
jdbc_url = f"jdbc:sqlserver://{server};databaseName={database};applicationIntent=ReadOnly"

# Connection properties
connection_properties = {
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    "user": username,
    "password": password,
    "applicationIntent": "ReadOnly"  # Ensure read-only access
}

:: loading settings :: file = /opt/spark/conf/ivy_settings.xml
:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.microsoft.sqlserver#mssql-jdbc added as a dependency
com.oracle.database.jdbc#ojdbc11 added as a dependency
com.databricks#spark-xml_2.12 added as a dependency
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-aaeeb6e5-30c8-4b77-9c28-efa47fcd4c24;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.2 in nexus-public
	found com.amazonaws#aws-java-sdk-bundle;1.11.1026 in nexus-public
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in nexus-public
	found com.microsoft.sqlserver#mssql-jdbc;12.2.0.jre11 in nexus-public
	found com.oracle.database.jdbc#ojdbc11;23.2.0.0 in nexus-public
	found com.databricks#spark-xml_2.12;0.18.0 in nexus-public
	found commons-io#commons-io;2.11.0 in nexus-public
	found org.glassfish.jaxb#txw2;3.0.2 in nexus-public
	fo

In [ ]:
partition_col = "PAT_PSEUDO_ID"

# Get patient IDs with records from MIN_QUARTER onwards (from both Abr 1 and Abr 2)
recent_patients_query = f"""(
    SELECT DISTINCT Pat_Pseudo_ID 
    FROM {schema}.{mapping_table_abr1} 
    WHERE DWH_ZEITRAUM >= {MIN_QUARTER}
    UNION
    SELECT DISTINCT Pat_Pseudo_ID 
    FROM {schema}.{mapping_table_abr2} 
    WHERE DWH_ZEITRAUM >= {MIN_QUARTER}
) AS recent_patients"""

# Get partition bounds for filtered patient IDs
bounds_query = f"""(
    SELECT MIN(pl.{partition_col}) AS min_id, MAX(pl.{partition_col}) AS max_id 
    FROM {schema}.{table} pl
    WHERE EXISTS (
        SELECT 1 FROM {schema}.{mapping_table_abr1} m1 
        WHERE m1.Pat_Pseudo_ID = pl.PAT_PSEUDO_ID AND m1.DWH_ZEITRAUM >= {MIN_QUARTER}
    ) OR EXISTS (
        SELECT 1 FROM {schema}.{mapping_table_abr2} m2 
        WHERE m2.Pat_Pseudo_ID = pl.PAT_PSEUDO_ID AND m2.DWH_ZEITRAUM >= {MIN_QUARTER}
    )
) AS bounds"""

bounds_df = spark.read.jdbc(
    url=jdbc_url,
    table=bounds_query,
    properties=connection_properties
)

bounds = bounds_df.first()
min_id = bounds["min_id"]
max_id = bounds["max_id"]

print(f"Data filter: Patients with DWH_ZEITRAUM >= {MIN_QUARTER}")
print(f"Partition bounds: {min_id} - {max_id}")


[Stage 0:>                                                          (0 + 1) / 1]

Partition bounds: 16 28187820


In [ ]:
# Columns to retrieve from PATIENTEN_LOOKUP
columns = ["EGKVERSICHERTENNUMMER", "VORNAME", "NACHNAME", "Geburtsdatum", "PLZ"]

num_partitions = 64  # tune based on cluster size

# Join PATIENTEN_LOOKUP with mapping tables to filter by time period
# Include patients from both Abr 1 and Abr 2 with records from MIN_QUARTER onwards
filtered_table = f"""(
    SELECT pl.PAT_PSEUDO_ID, pl.{', pl.'.join(columns)}
    FROM {schema}.{table} pl
    WHERE EXISTS (
        SELECT 1 FROM {schema}.{mapping_table_abr1} m1 
        WHERE m1.Pat_Pseudo_ID = pl.PAT_PSEUDO_ID AND m1.DWH_ZEITRAUM >= {MIN_QUARTER}
    ) OR EXISTS (
        SELECT 1 FROM {schema}.{mapping_table_abr2} m2 
        WHERE m2.Pat_Pseudo_ID = pl.PAT_PSEUDO_ID AND m2.DWH_ZEITRAUM >= {MIN_QUARTER}
    )
) AS filtered_data"""

spark_df = spark.read.jdbc(
    url=jdbc_url,
    table=filtered_table,
    column=partition_col,  # PAT_PSEUDO_ID (numeric)
    lowerBound=int(min_id),
    upperBound=int(max_id),
    numPartitions=num_partitions,
    properties=connection_properties
).drop("PAT_PSEUDO_ID")  # Drop after partitioning, not needed for analysis

total_rows = spark_df.count()
print(f"Loaded {total_rows:,} patient variants from {schema}.{table}")
print(f"Filter: Patients with records from DWH_ZEITRAUM >= {MIN_QUARTER} (Abr 1 + Abr 2)")

[Stage 1:==============================================>          (52 + 9) / 64]

Loaded 30,541,347 rows from dbo.M20_HIS_T_01AB000_PATIENTEN_LOOKUP


## 2. General Dataset Statistics

This section provides an overview of the dataset including size, completeness, and basic field-level statistics.


In [ ]:
# Calculate basic dataset statistics
completeness_stats = spark_df.select([
    F.count(F.lit(1)).alias("total"),
    F.sum(F.when(F.col("EGKVERSICHERTENNUMMER").isNotNull(), 1).otherwise(0)).alias("kvnr_count"),
    F.sum(F.when(F.col("VORNAME").isNotNull() & (F.col("VORNAME") != ""), 1).otherwise(0)).alias("vorname_count"),
    F.sum(F.when(F.col("NACHNAME").isNotNull() & (F.col("NACHNAME") != ""), 1).otherwise(0)).alias("nachname_count"),
    F.sum(F.when(F.col("Geburtsdatum").isNotNull(), 1).otherwise(0)).alias("geburtsdatum_count"),
    F.sum(F.when(F.col("PLZ").isNotNull(), 1).otherwise(0)).alias("plz_count")
]).first()

completeness_df = pd.DataFrame({
    "Field": ["EGKVERSICHERTENNUMMER", "VORNAME", "NACHNAME", "Geburtsdatum", "PLZ"],
    "Non_Null_Count": [
        completeness_stats["kvnr_count"],
        completeness_stats["vorname_count"],
        completeness_stats["nachname_count"],
        completeness_stats["geburtsdatum_count"],
        completeness_stats["plz_count"]
    ]
})
completeness_df["Completeness_Pct"] = (completeness_df["Non_Null_Count"] / total_rows * 100).round(2)
completeness_df["Missing_Pct"] = 100 - completeness_df["Completeness_Pct"]

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nTotal Records: {total_rows:,}")
print(f"Fields Analyzed: 5")
print("\nField Completeness:")
print(completeness_df.to_string(index=False))
print("\n" + "=" * 60)


## 3. Privacy-Preserving Statistical Analysis

This section extracts **aggregated, non-identifying statistics** for generating realistic synthetic data while maintaining privacy. All analyses use aggregated counts and distributions - never individual records.

### Privacy Principles Applied

1. **k-Anonymity**: Only show value frequencies where count ≥ 100
2. **Aggregation Only**: No individual records, only statistical summaries
3. **Binning**: Sensitive values (dates, PLZs) are binned into ranges
4. **Count Rounding**: All counts rounded to nearest 100


### 3.1 Name Frequency Distributions (k≥100)

Extract frequency distributions of first and last names, applying k-anonymity.

**Privacy**: Safe - shows only common names appearing 100+ times.


In [ ]:
def extract_name_distribution(df, name_col: str, k: int = K_ANONYMITY, top_n: int = 1000) -> pd.DataFrame:
    """Extract k-anonymized name frequency distribution."""
    return (
        df.filter(F.col(name_col).isNotNull() & (F.col(name_col) != ""))
        .groupBy(name_col)
        .count()
        .filter(F.col("count") >= k)
        .orderBy(F.desc("count"))
        .limit(top_n)
        .withColumn("frequency_pct", F.round((F.col("count") / total_rows) * 100, 4))
        .toPandas()
    )

vorname_dist = extract_name_distribution(spark_df, "VORNAME", k=K_ANONYMITY)
nachname_dist = extract_name_distribution(spark_df, "NACHNAME", k=K_ANONYMITY)

print(f"VORNAME distribution (k≥{K_ANONYMITY}): {len(vorname_dist)} unique names")
print(f"Coverage: {vorname_dist['count'].sum():,} records ({vorname_dist['count'].sum()/total_rows*100:.2f}%)")
print("\nTop 20 first names:")
print(vorname_dist.head(20).to_string(index=False))

print(f"\n\nNACHNAME distribution (k≥{K_ANONYMITY}): {len(nachname_dist)} unique names")
print(f"Coverage: {nachname_dist['count'].sum():,} records ({nachname_dist['count'].sum()/total_rows*100:.2f}%)")
print("\nTop 20 last names:")
print(nachname_dist.head(20).to_string(index=False))


### 3.2 Name Length Distributions

**Privacy**: Safe - no actual names, only length statistics.


In [ ]:
def extract_length_distribution(df, col_name: str) -> pd.DataFrame:
    """Extract length distribution for a string column."""
    return (
        df.filter(F.col(col_name).isNotNull() & (F.col(col_name) != ""))
        .withColumn("length", F.length(F.col(col_name)))
        .groupBy("length")
        .count()
        .withColumn("frequency_pct", F.round((F.col("count") / total_rows) * 100, 4))
        .orderBy("length")
        .toPandas()
    )

vorname_lengths = extract_length_distribution(spark_df, "VORNAME")
nachname_lengths = extract_length_distribution(spark_df, "NACHNAME")

print("VORNAME Length Distribution:")
print(vorname_lengths.to_string(index=False))
print("\n\nNACHNAME Length Distribution:")
print(nachname_lengths.to_string(index=False))


### 3.2b Name Character Patterns

**Privacy**: Safe - only character-level statistics, no actual names.


In [ ]:
# Name character patterns: special characters and patterns
def analyze_name_patterns(df, col_name: str) -> dict:
    """Analyze character patterns in names."""
    valid_names = df.filter(F.col(col_name).isNotNull() & (F.col(col_name) != ""))
    total = valid_names.count()
    
    patterns = valid_names.select(
        # Contains hyphen (compound names like "Anna-Marie")
        F.sum(F.when(F.col(col_name).contains("-"), 1).otherwise(0)).alias("hyphenated"),
        # Contains space (multi-part like "Maria Clara")
        F.sum(F.when(F.col(col_name).contains(" "), 1).otherwise(0)).alias("with_space"),
        # Contains umlauts (ä, ö, ü, Ä, Ö, Ü)
        F.sum(F.when(F.col(col_name).rlike("[äöüÄÖÜ]"), 1).otherwise(0)).alias("with_umlaut"),
        # Contains ß
        F.sum(F.when(F.col(col_name).contains("ß"), 1).otherwise(0)).alias("with_eszett")
    ).first()
    
    return {
        "total": total,
        "hyphenated": patterns["hyphenated"] or 0,
        "with_space": patterns["with_space"] or 0,
        "with_umlaut": patterns["with_umlaut"] or 0,
        "with_eszett": patterns["with_eszett"] or 0
    }

vorname_patterns = analyze_name_patterns(spark_df, "VORNAME")
nachname_patterns = analyze_name_patterns(spark_df, "NACHNAME")

print("VORNAME Character Patterns:")
for k, v in vorname_patterns.items():
    if k != "total":
        pct = v / vorname_patterns["total"] * 100
        print(f"  {k}: {v:,} ({pct:.2f}%)")

print("\nNACHNAME Character Patterns:")
for k, v in nachname_patterns.items():
    if k != "total":
        pct = v / nachname_patterns["total"] * 100
        print(f"  {k}: {v:,} ({pct:.2f}%)")


### 3.3 KVNR Character Patterns

**Privacy**: Safe - only letter/digit distributions, not actual KVNRs.


In [ ]:
# KVNR first letter distribution (only valid 10-char KVNRs)
valid_kvnrs = spark_df.filter(
    F.col("EGKVERSICHERTENNUMMER").isNotNull() & 
    (F.length(F.col("EGKVERSICHERTENNUMMER")) == 10)
)

letter_dist = (
    valid_kvnrs
    .withColumn("first_letter", F.substring(F.col("EGKVERSICHERTENNUMMER"), 1, 1))
    .groupBy("first_letter")
    .count()
    .withColumn("frequency_pct", F.round((F.col("count") / total_rows) * 100, 4))
    .orderBy(F.desc("count"))
    .toPandas()
)

print("KVNR First Letter Distribution:")
print(letter_dist.to_string(index=False))


### 3.4 PLZ Regional Distribution

**Privacy**: Safe - only regional aggregates (first 2 digits = German regions), no individual postal codes.


In [ ]:
# PLZ regional distribution (first 2 digits = German regions)
# Privacy: No individual PLZs, only regional aggregates
valid_plzs = spark_df.filter(F.col("PLZ").isNotNull()).withColumn(
    "plz_str", F.col("PLZ").cast(StringType())
).filter(F.length(F.col("plz_str")) == 5)

# Regional distribution (first 2 digits)
plz_regional = (
    valid_plzs
    .withColumn("plz_region", F.substring(F.col("plz_str"), 1, 2))
    .groupBy("plz_region")
    .count()
    .withColumn("frequency_pct", F.round((F.col("count") / total_rows) * 100, 2))
    .orderBy("plz_region")
    .toPandas()
)

# Also get statistics on PLZ numeric values (without revealing individual PLZs)
plz_int_col = F.col("PLZ").cast(IntegerType())
plz_stats = spark_df.filter(F.col("PLZ").isNotNull()).select(
    F.min(plz_int_col).alias("min_plz"),
    F.max(plz_int_col).alias("max_plz"),
    F.round(F.avg(plz_int_col), 0).alias("mean_plz"),
    F.round(F.stddev(plz_int_col), 0).alias("std_plz")
).first()

print(f"PLZ Regional Distribution ({len(plz_regional)} regions):")
print(plz_regional.to_string(index=False))
print(f"\nPLZ Statistics: min={plz_stats['min_plz']}, max={plz_stats['max_plz']}, "
      f"mean≈{int(plz_stats['mean_plz'])}, std≈{int(plz_stats['std_plz'])}")


### 3.5 Age/Birth Year Distribution (Binned)

**Privacy**: Safe - binned dates, not individual birth dates.


In [ ]:
# Birth year distribution (5-year bins)
valid_dates = (
    spark_df.filter(F.col("Geburtsdatum").isNotNull())
    .withColumn("birth_year", F.year(F.to_date(F.col("Geburtsdatum"), "yyyy-MM-dd")))
    .filter(F.col("birth_year").isNotNull())
    .withColumn("birth_year_bin", (F.col("birth_year") / 5).cast(IntegerType()) * 5)
)

birth_year_dist = (
    valid_dates
    .groupBy("birth_year_bin")
    .count()
    .withColumn("frequency_pct", F.round((F.col("count") / total_rows) * 100, 4))
    .orderBy("birth_year_bin")
    .toPandas()
)

# Age statistics
current_year = datetime.now().year
age_stats = (
    valid_dates
    .withColumn("age", F.lit(current_year) - F.col("birth_year"))
    .select(
        F.min("age").alias("min_age"),
        F.max("age").alias("max_age"),
        F.avg("age").alias("mean_age"),
        F.stddev("age").alias("std_age")
    )
    .first()
)

print("Birth Year Distribution (5-year bins):")
print(birth_year_dist.to_string(index=False))
print(f"\n\nAge Statistics:")
print(f"Min: {age_stats['min_age']}, Max: {age_stats['max_age']}")
print(f"Mean: {age_stats['mean_age']:.1f}, Std Dev: {age_stats['std_age']:.1f}")


### 3.6 Birth Date Patterns (Month/Day)

**Privacy**: Safe - only aggregate month and day-of-month distributions.


In [ ]:
# Month and day-of-month distributions
birth_dates = spark_df.filter(F.col("Geburtsdatum").isNotNull()).withColumn(
    "birth_date", F.to_date(F.col("Geburtsdatum"), "yyyy-MM-dd")
).filter(F.col("birth_date").isNotNull())

# Month distribution (1-12)
month_dist = (
    birth_dates
    .withColumn("month", F.month(F.col("birth_date")))
    .groupBy("month")
    .count()
    .withColumn("frequency_pct", F.round((F.col("count") / total_rows) * 100, 2))
    .orderBy("month")
    .toPandas()
)

# Day-of-month distribution (1-31)
day_dist = (
    birth_dates
    .withColumn("day", F.dayofmonth(F.col("birth_date")))
    .groupBy("day")
    .count()
    .withColumn("frequency_pct", F.round((F.col("count") / total_rows) * 100, 2))
    .orderBy("day")
    .toPandas()
)

print("Birth Month Distribution:")
print(month_dist.to_string(index=False))
print("\n\nBirth Day-of-Month Distribution:")
print(day_dist.to_string(index=False))


### 3.7 Gender Distribution

**Privacy**: Safe - only aggregate counts for a small set of values.

**Note**: Gender data comes from `M20_HIS_T_01AB100_SCHEINE` table, filtered by `MIN_QUARTER`.


In [ ]:
# Load gender distribution from SCHEINE table (filtered by MIN_QUARTER)
scheine_table = "M20_HIS_T_01AB100_SCHEINE"

# Query with time filter
gender_query = f"""(
    SELECT GESCHLECHT
    FROM {schema}.{scheine_table}
    WHERE DWH_ZEITRAUM >= {MIN_QUARTER}
) AS gender_data"""

scheine_df = spark.read.jdbc(
    url=jdbc_url,
    table=gender_query,
    numPartitions=64,
    properties=connection_properties
)

scheine_total = scheine_df.count()
print(f"SCHEINE records loaded (DWH_ZEITRAUM >= {MIN_QUARTER}): {scheine_total:,}")


In [ ]:
# Gender distribution
gender_dist = (
    scheine_df.filter(F.col("GESCHLECHT").isNotNull())
    .groupBy("GESCHLECHT")
    .count()
    .withColumn("frequency_pct", F.round((F.col("count") / scheine_total) * 100, 4))
    .orderBy(F.desc("count"))
    .toPandas()
)

print("Gender Distribution (from SCHEINE table):")
print(gender_dist.to_string(index=False))
print(f"\nTotal with gender: {gender_dist['count'].sum():,} ({gender_dist['count'].sum()/scheine_total*100:.2f}%)")


### 3.8 VSDM Verification Distribution

**Privacy**: Safe - only aggregate counts of verification outcomes.

VSDM (Versichertenstammdatenmanagement) verification data is used in survivorship to identify the most trustworthy patient records. Data filtered by `MIN_QUARTER`.


In [ ]:
# Load VSDM verification data with quarter information (filtered by MIN_QUARTER)
# VSDM is joined with SCHEINE to get DWH_ZEITRAUM (quarter)
# Note: Both VSDM and SCHEINE can have multiple records per SCHEINID
vsdm_table = "M20_HIS_T_01AB100_VSDMPRUEFUNGSNACHWEISE"
scheine_table = "M20_HIS_T_01AB100_SCHEINE"

# Query to join VSDM with SCHEINE, filtered by time
# Keep one row per (SCHEINID, ERGEBNISONLINEPRUEFUNG, ERRORCODE) for distribution analysis
vsdm_query = f"""(
    SELECT DISTINCT
        v.SCHEINID,
        v.ERGEBNISONLINEPRUEFUNG,
        v.ERRORCODE,
        s.DWH_ZEITRAUM
    FROM {schema}.{vsdm_table} v
    INNER JOIN (
        SELECT DISTINCT SCHEINID, DWH_ZEITRAUM 
        FROM {schema}.{scheine_table} 
        WHERE DWH_ZEITRAUM >= {MIN_QUARTER}
    ) s ON v.SCHEINID = s.SCHEINID
) AS vsdm_with_quarter"""

vsdm_df = spark.read.jdbc(
    url=jdbc_url,
    table=vsdm_query,
    properties=connection_properties
)

# For coverage: count distinct SCHEINIDs with VSDM (not total VSDM records)
vsdm_scheine_count = vsdm_df.select("SCHEINID").distinct().count()
print(f"SCHEINIDs with VSDM verification (DWH_ZEITRAUM >= {MIN_QUARTER}): {vsdm_scheine_count:,}")

# Count distinct SCHEINIDs in SCHEINE
scheine_total = spark.read.jdbc(
    url=jdbc_url,
    table=f"(SELECT COUNT(DISTINCT SCHEINID) as cnt FROM {schema}.{scheine_table} WHERE DWH_ZEITRAUM >= {MIN_QUARTER}) AS scheine_count",
    properties=connection_properties
).first()["cnt"]
print(f"Total distinct SCHEINIDs (DWH_ZEITRAUM >= {MIN_QUARTER}): {scheine_total:,}")
print(f"Overall VSDM coverage: {vsdm_scheine_count/scheine_total*100:.2f}%")

# Also store total VSDM records for later distribution analysis
vsdm_total = vsdm_df.count()
print(f"Total VSDM records for distribution analysis: {vsdm_total:,}")


In [ ]:
# VSDM verification result distribution
vsdm_result_dist = (
    vsdm_df
    .groupBy("ERGEBNISONLINEPRUEFUNG")
    .count()
    .withColumn("frequency_pct", F.round((F.col("count") / vsdm_total) * 100, 4))
    .orderBy("ERGEBNISONLINEPRUEFUNG")
    .toPandas()
)

print("VSDM Verification Result Distribution (ERGEBNISONLINEPRUEFUNG):")
print(vsdm_result_dist.to_string(index=False))

# Error code distribution (for records with errors)
vsdm_error_dist = (
    vsdm_df
    .filter(F.col("ERRORCODE").isNotNull() & (F.col("ERRORCODE") != 0))
    .groupBy("ERRORCODE")
    .count()
    .withColumn("frequency_pct", F.round((F.col("count") / vsdm_total) * 100, 4))
    .orderBy(F.desc("count"))
    .limit(20)
    .toPandas()
)

print("\nVSDM Error Code Distribution (top 20):")
print(vsdm_error_dist.to_string(index=False))

# Calculate valid VSDM rate (among records that HAVE VSDM verification)
valid_vsdm_count = vsdm_df.filter(
    (F.col("ERGEBNISONLINEPRUEFUNG").isin([1, 2])) & 
    ((F.col("ERRORCODE").isNull()) | (F.col("ERRORCODE") == 0))
).count()

vsdm_valid_rate = valid_vsdm_count / vsdm_total
print(f"\nValid VSDM verifications (Tier 1 eligible): {valid_vsdm_count:,} ({vsdm_valid_rate*100:.2f}%)")
print("  Note: This is rate AMONG records with VSDM, not overall population")

# Overall VSDM coverage rate (using distinct SCHEINID counts)
vsdm_coverage_rate = vsdm_scheine_count / scheine_total
print(f"\nOverall VSDM coverage: {vsdm_coverage_rate*100:.2f}% of all SCHEINIDs")


In [ ]:
# === TEMPORAL VSDM ANALYSIS ===
# VSDM verification was introduced in recent years - older records don't have it
# This analysis shows VSDM coverage by quarter (filtered by MIN_QUARTER)

# Get distinct SCHEINE count by quarter (filtered by MIN_QUARTER)
# Use COUNT(DISTINCT SCHEINID) since SCHEINE has composite key
scheine_by_quarter = spark.read.jdbc(
    url=jdbc_url,
    table=f"(SELECT DWH_ZEITRAUM, COUNT(DISTINCT SCHEINID) as total_records FROM {schema}.{scheine_table} WHERE DWH_ZEITRAUM >= {MIN_QUARTER} GROUP BY DWH_ZEITRAUM) AS q",
    properties=connection_properties
).toPandas()

# Get distinct VSDM SCHEINID count by quarter (for coverage calculation)
vsdm_by_quarter = (
    vsdm_df
    .groupBy("DWH_ZEITRAUM")
    .agg(F.countDistinct("SCHEINID").alias("vsdm_scheine"))
    .toPandas()
)

# Merge to calculate coverage by quarter
temporal_vsdm = scheine_by_quarter.merge(vsdm_by_quarter, on="DWH_ZEITRAUM", how="left")
temporal_vsdm["vsdm_scheine"] = temporal_vsdm["vsdm_scheine"].fillna(0).astype(int)
temporal_vsdm["coverage_pct"] = (temporal_vsdm["vsdm_scheine"] / temporal_vsdm["total_records"] * 100).round(2)
temporal_vsdm = temporal_vsdm.sort_values("DWH_ZEITRAUM")

print("VSDM Coverage by Quarter (DWH_ZEITRAUM):")
print("=" * 70)
print(temporal_vsdm.to_string(index=False))

# Identify when VSDM started (first quarter with >10% coverage)
vsdm_start_quarter = temporal_vsdm[temporal_vsdm["coverage_pct"] > 10]["DWH_ZEITRAUM"].min()
print(f"\nVSDM started appearing significantly from quarter: {vsdm_start_quarter}")

# Calculate year-based coverage for simpler synthesis
temporal_vsdm["year"] = (temporal_vsdm["DWH_ZEITRAUM"] // 10).astype(int)
yearly_coverage = temporal_vsdm.groupby("year").agg({
    "total_records": "sum",
    "vsdm_scheine": "sum"
}).reset_index()
yearly_coverage["coverage_pct"] = (yearly_coverage["vsdm_scheine"] / yearly_coverage["total_records"] * 100).round(2)

print("\nVSDM Coverage by Year:")
print(yearly_coverage.to_string(index=False))


### 3.7 Cardinality & Uniqueness Statistics

**Privacy**: Safe - only aggregate counts and ratios.


In [ ]:
# Cardinality: unique values per field
columns = ["EGKVERSICHERTENNUMMER", "VORNAME", "NACHNAME", "Geburtsdatum", "PLZ"]

cardinality_data = []
for col in columns:
    stats = spark_df.agg(
        F.countDistinct(F.col(col)).alias("unique"),
        F.count(F.when(F.col(col).isNotNull(), 1)).alias("non_null")
    ).first()
    cardinality_data.append({
        "field": col,
        "unique_values": stats["unique"],
        "non_null_count": stats["non_null"],
        "uniqueness_ratio": round(stats["unique"] / stats["non_null"], 4) if stats["non_null"] > 0 else 0
    })

cardinality_df = pd.DataFrame(cardinality_data)
print("Field Cardinality & Uniqueness:")
print(cardinality_df.to_string(index=False))


## 4. Data Quality Analysis

This section applies validation rules to detect errors, then analyzes error patterns and distributions.

### 4.1 Error Detection

Apply validation rules (defined in Section 1) to classify each field value as valid or invalid.

In [ ]:
validated_df = spark_df.withColumns({
    "kvnr_error": validate_kvnr_udf(F.col("EGKVERSICHERTENNUMMER")), # Standard UDF (Python)
    "vorname_error": get_name_check_expr("VORNAME"),                 # Native (JVM)
    "nachname_error": get_name_check_expr("NACHNAME"),               # Native (JVM)
    "geburtsdatum_error": get_date_check_expr("Geburtsdatum"),       # Native (JVM)
    "plz_error": get_plz_check_expr("PLZ")                           # Native (JVM)
})

In [ ]:
# Aggregate error statistics in a single pass
# These statistics will be used to determine error injection probabilities for synthetic data
agg_row = validated_df.agg(
    # Per-column error counts
    F.sum(F.when(F.col("kvnr_error").isNotNull(), 1).otherwise(0)).alias("kvnr_errors"),
    F.sum(F.when(F.col("vorname_error").isNotNull(), 1).otherwise(0)).alias("vorname_errors"),
    F.sum(F.when(F.col("nachname_error").isNotNull(), 1).otherwise(0)).alias("nachname_errors"),
    F.sum(F.when(F.col("geburtsdatum_error").isNotNull(), 1).otherwise(0)).alias("geburtsdatum_errors"),
    F.sum(F.when(F.col("plz_error").isNotNull(), 1).otherwise(0)).alias("plz_errors"),

    # Rows with at least one error
    F.sum(
        F.when(
            F.col("kvnr_error").isNotNull() |
            F.col("vorname_error").isNotNull() |
            F.col("nachname_error").isNotNull() |
            F.col("geburtsdatum_error").isNotNull() |
            F.col("plz_error").isNotNull(),
            1
        ).otherwise(0)
    ).alias("rows_with_errors"),

    # Total rows
    F.count(F.lit(1)).alias("total_rows")
).first()


In [ ]:
# Extract scalar values on the driver
rows_with_errors  = agg_row["rows_with_errors"]

kvnr_errors       = agg_row["kvnr_errors"]
vorname_errors    = agg_row["vorname_errors"]
nachname_errors   = agg_row["nachname_errors"]
geburtsdatum_errs = agg_row["geburtsdatum_errors"]
plz_errors        = agg_row["plz_errors"]

# Build tiny pandas DataFrame on driver (5 rows => negligible mem)
stats_data = {
    "Column": [
        "EGKVERSICHERTENNUMMER",
        "VORNAME",
        "NACHNAME",
        "Geburtsdatum",
        "PLZ"
    ],
    "Error_Count": [
        kvnr_errors,
        vorname_errors,
        nachname_errors,
        geburtsdatum_errs,
        plz_errors
    ]
}

stats_df = pd.DataFrame(stats_data)
stats_df["Error_Percentage"] = (stats_df["Error_Count"] / total_rows * 100).round(2)
stats_df["Valid_Percentage"] = (100 - stats_df["Error_Percentage"]).round(2)
stats_df["Error_Count_Rounded"] = stats_df["Error_Count"].apply(round_count)
stats_df = stats_df.sort_values("Error_Count", ascending=False)

# Display results (counts rounded to nearest 100 for privacy)
print("\n" + "="*60)
print("VALIDATION STATISTICS REPORT (counts rounded to nearest 100)")
print("="*60)
print(f"\nTotal rows processed: ~{round_count(total_rows):,}")
print(f"Rows with at least one error: ~{round_count(rows_with_errors):,} ({(rows_with_errors/total_rows*100):.2f}%)")
print(f"Rows with no errors: ~{round_count(total_rows - rows_with_errors):,} "
      f"({((total_rows - rows_with_errors)/total_rows*100):.2f}%)")

total_field_errors = stats_df["Error_Count"].sum()
print(f"\nTotal field errors: ~{round_count(int(total_field_errors)):,}")

print("\n" + "-"*60)
print("Error Statistics by Column:")
print("-"*60)
# Display with rounded counts only
display_df = stats_df[["Column", "Error_Count_Rounded", "Error_Percentage", "Valid_Percentage"]].copy()
display_df.columns = ["Column", "Error_Count (~)", "Error_%", "Valid_%"]
display(display_df) if 'display' in locals() else print(display_df.to_string(index=False))

print("\n" + "="*60)


VALIDATION STATISTICS REPORT

Total rows processed: 30,541,347
Rows with at least one error: 6,605,528 (21.63%)
Rows with no errors: 23,935,819 (78.37%)

Total field errors: 6,973,086

------------------------------------------------------------
Error Statistics by Column:
------------------------------------------------------------
               Column  Error_Count  Error_Percentage  Valid_Percentage
EGKVERSICHERTENNUMMER      6472276             21.19             78.81
             NACHNAME       246804              0.81             99.19
                  PLZ       128248              0.42             99.58
         Geburtsdatum        82647              0.27             99.73
              VORNAME        43111              0.14             99.86



### 4.2 Error Type Distribution

Analyze the distribution of specific error types for each field — this tells us which errors to inject and how frequently.

These distributions will be used to generate synthetic errors with the same characteristics.

In [ ]:
# The columns containing our error messages
error_columns = [
    "kvnr_error", 
    "vorname_error", 
    "nachname_error", 
    "geburtsdatum_error", 
    "plz_error"
]

error_distributions = {}

print("Fetching error distributions...")

for col_name in error_columns:
    # 1. Filter to get only rows with errors
    # 2. Group by the specific error message
    # 3. Count occurrences
    # 4. Order by most frequent
    counts_df = validated_df.filter(F.col(col_name).isNotNull()) \
                            .groupBy(col_name) \
                            .count() \
                            .withColumnRenamed("count", "frequency") \
                            .withColumn("percentage", F.round((F.col("frequency") / total_rows) * 100, 4)) \
                            .orderBy(F.desc("frequency"))
    
    # Collect the tiny summary table to Pandas (Driver)
    # This is safe because there are only a handful of distinct error messages per column.
    error_distributions[col_name] = counts_df.toPandas()
    
    print(f"--- {col_name} ---")
    print(error_distributions[col_name])
    print("\n")

Fetching error distributions...


--- kvnr_error ---
                               kvnr_error  frequency  percentage
0             Missing or non-string input    6210529     20.3348
1   Checksum mismatch (Expected 0, Got 9)      10822      0.0354
2   Checksum mismatch (Expected 2, Got 9)       7751      0.0254
3   Checksum mismatch (Expected 2, Got 0)       5521      0.0181
4   Checksum mismatch (Expected 3, Got 0)       4496      0.0147
..                                    ...        ...         ...
86  Checksum mismatch (Expected 2, Got 7)       1502      0.0049
87  Checksum mismatch (Expected 7, Got 2)       1473      0.0048
88  Checksum mismatch (Expected 0, Got 5)       1446      0.0047
89  Checksum mismatch (Expected 9, Got 4)       1407      0.0046
90  Checksum mismatch (Expected 1, Got 6)       1329      0.0044

[91 rows x 3 columns]




--- vorname_error ---
                      vorname_error  frequency  percentage
0  Name contains invalid characters      43111      0.1412




--- nachname_error ---
                     nachname_error  frequency  percentage
0  Name contains invalid characters     246804      0.8081




--- geburtsdatum_error ---
        geburtsdatum_error  frequency  percentage
0  Date is > 120 years ago      80787      0.2645
1             Missing date       1860      0.0061




--- plz_error ---
                                         plz_error  frequency  percentage
0                   PLZ out of valid numeric range      76170      0.2494
1                                      Missing PLZ       4968      0.0163
2         Invalid format: 4730 (expected 5 digits)        492      0.0016
3         Invalid format: 6291 (expected 5 digits)        488      0.0016
4         Invalid format: 4720 (expected 5 digits)        362      0.0012
...                                            ...        ...         ...
18165   Invalid format: 13-203 (expected 5 digits)          1      0.0000
18166   Invalid format: 5616LH (expected 5 digits)          1      0.0000
18167   Invalid format: 107590 (expected 5 digits)          1      0.0000
18168  Invalid format: 6602 GS (expected 5 digits)          1      0.0000
18169   Invalid format: 93-649 (expected 5 digits)          1      0.0000

[18170 rows x 3 columns]




### 4.3 Error Profile Summary

Compile error statistics into a structured format for synthetic data generation.


In [ ]:
# Compile error statistics for synthesis
error_profile = {
    "total_rows": total_rows,
    "rows_with_errors": rows_with_errors,
    "global_error_rate": rows_with_errors / total_rows,
    "field_errors": {}
}

# Build field-level error stats from the error_distributions dict
for col_name, dist_df in error_distributions.items():
    field_name = col_name.replace("_error", "").upper()
    total_errors = dist_df["frequency"].sum()
    error_rate = total_errors / total_rows
    
    # Get error type breakdown
    error_types = {}
    for _, row in dist_df.iterrows():
        error_msg = row[col_name]
        error_types[error_msg] = {
            "count": int(row["frequency"]),
            "rate": row["frequency"] / total_rows
        }
    
    error_profile["field_errors"][field_name] = {
        "total_errors": int(total_errors),
        "error_rate": error_rate,
        "error_types": error_types
    }

# Display summary (counts rounded for privacy)
print("=" * 70)
print("ERROR PROFILE FOR SYNTHETIC DATA GENERATION (counts ~rounded)")
print("=" * 70)
print(f"\nDataset Size: ~{round_count(total_rows):,} rows")
print(f"Global Error Rate: {error_profile['global_error_rate']*100:.2f}%")
print(f"Rows with Errors: ~{round_count(rows_with_errors):,}")
print("\nField-Level Error Rates:")
print("-" * 70)

for field, stats in sorted(error_profile["field_errors"].items(), 
                           key=lambda x: x[1]["error_rate"], reverse=True):
    print(f"\n{field}: {stats['error_rate']*100:.2f}% error rate (~{round_count(stats['total_errors']):,} errors)")
    for error_type, type_stats in list(stats["error_types"].items())[:3]:
        print(f"  → {error_type[:50]}: {type_stats['rate']*100:.4f}%")


### 4.4 Error Correlations

Analyze whether errors in one field are correlated with errors in other fields — useful for realistic error injection.


In [ ]:
# Error correlations: P(error_B | error_A)
# This tells us: if field A has an error, how likely is field B to also have an error?
error_cols = ["kvnr_error", "vorname_error", "nachname_error", "geburtsdatum_error", "plz_error"]

# Create binary error flags
error_flags_df = validated_df.select(
    *[F.when(F.col(c).isNotNull(), 1).otherwise(0).alias(c.replace("_error", "_has_error"))
      for c in error_cols]
)

# Calculate conditional probabilities
error_correlations = {}
for col_a in error_cols:
    flag_a = col_a.replace("_error", "_has_error")
    name_a = col_a.replace("_error", "").upper()
    
    # Count records where A has an error
    rows_with_a = error_flags_df.filter(F.col(flag_a) == 1)
    count_a = rows_with_a.count()
    
    if count_a > 0:
        error_correlations[name_a] = {"count": count_a, "conditional_probs": {}}
        for col_b in error_cols:
            if col_a != col_b:
                flag_b = col_b.replace("_error", "_has_error")
                name_b = col_b.replace("_error", "").upper()
                # P(error_B | error_A) = count(A AND B) / count(A)
                count_ab = rows_with_a.filter(F.col(flag_b) == 1).count()
                prob = count_ab / count_a
                error_correlations[name_a]["conditional_probs"][name_b] = round(prob, 4)

# Display as a matrix-like summary
print("Error Correlations: P(column | row has error)")
print("=" * 70)
print(f"{'Field':20} | ", end="")
print(" | ".join(f"{c.replace('_error', '').upper():>10}" for c in error_cols))
print("-" * 70)

for field_a, data in error_correlations.items():
    row = [f"{field_a:20} |"]
    for col_b in error_cols:
        name_b = col_b.replace("_error", "").upper()
        if name_b == field_a:
            row.append(f"{'---':>10}")
        else:
            prob = data["conditional_probs"].get(name_b, 0)
            row.append(f"{prob*100:>9.2f}%")
    print(" | ".join(row))


## 5. Business Rule Analysis

Analyze duplicate KVNR patterns to identify systematic placeholders and integrity violations.


In [ ]:
def analyze_duplicate_kvnrs(
    df, 
    kvnr_col: str = "EGKVERSICHERTENNUMMER",
    manual_blocklist: Optional[list[str]] = None
) -> tuple:
    """
    Analyze duplicate KVNRs and categorize them.
    
    Args:
        df: Spark DataFrame with patient data
        kvnr_col: Name of KVNR column
        manual_blocklist: List of known test/dummy KVNRs to exclude
        
    Returns:
        Tuple of (placeholders_df, integrity_df) DataFrames
    """
    if manual_blocklist is None:
        manual_blocklist = [
            "X000000000", "M999999999", "A999999994", "A123456780", 
            "S000000000", "H999999999", "E123456789"
        ]
    
    # Find all duplicates (count > 1)
    all_duplicates = (
        df.filter(F.col(kvnr_col).isNotNull())
        .groupBy(kvnr_col)
        .count()
        .filter(F.col("count") > 1)
        .cache()
    )
    
    # Apply technical validation
    analyzed_duplicates = (
        all_duplicates
        .withColumn("validation_error", validate_kvnr_udf(kvnr_col))
        .withColumn("is_valid", F.col("validation_error").isNull())
    )
    
    # Category A: Systematic Placeholders (invalid or in blocklist)
    placeholders_df = (
        analyzed_duplicates
        .filter(
            (F.col("is_valid") == False) | 
            (F.col(kvnr_col).isin(manual_blocklist))
        )
        .orderBy(F.col("count").desc())
    )
    
    # Category B: Integrity Violations (valid but shared)
    integrity_df = (
        analyzed_duplicates
        .filter(
            (F.col("is_valid") == True) & 
            (~F.col(kvnr_col).isin(manual_blocklist))
        )
        .orderBy(F.col("count").desc())
    )
    
    return placeholders_df, integrity_df


# Execute duplicate analysis
manual_dummies = [
    "X000000000", "M999999999", "A999999994", "A123456780", 
    "S000000000", "H999999999", "E123456789"
]

placeholders_df, integrity_df = analyze_duplicate_kvnrs(
    spark_df, 
    manual_blocklist=manual_dummies
)

25/11/28 19:58:54 WARN CacheManager: Asked to cache already cached data.


In [ ]:
def print_duplicate_report(
    placeholders_df,
    integrity_df,
    total_rows: int,
    kvnr_col: str = "EGKVERSICHERTENNUMMER"
) -> None:
    """
    Print formatted report of duplicate KVNR analysis.
    
    Args:
        placeholders_df: DataFrame with placeholder duplicates
        integrity_df: DataFrame with integrity violation duplicates
        total_rows: Total number of rows in dataset
        kvnr_col: Name of KVNR column
    """
    # Calculate statistics
    placeholder_rows = placeholders_df.agg(F.sum("count")).collect()[0][0] or 0
    placeholder_pct = (placeholder_rows / total_rows) * 100
    
    integrity_rows = integrity_df.agg(F.sum("count")).collect()[0][0] or 0
    integrity_pct = (integrity_rows / total_rows) * 100
    
    # Print report (counts rounded for privacy)
    print("=" * 60)
    print("BUSINESS ERROR ANALYSIS: DUPLICATE KVNRs (counts ~rounded)")
    print("=" * 60)
    print(f"Total Rows Processed: ~{round_count(total_rows):,}\n")
    
    print("--- 1. Systematic Placeholders (Invalid & Shared) ---")
    print(f"Affected Rows: ~{round_count(int(placeholder_rows)):,} ({placeholder_pct:.4f}%)")
    print(f"Unique Variants: ~{round_count(placeholders_df.count()):,}")
    print("Most common placeholders:")
    placeholders_df.select(kvnr_col, "count", "validation_error").show(
        20, truncate=False
    )
    
    print("\n--- 2. Integrity Violations (Valid & Shared) ---")
    print(f"Affected Rows: ~{round_count(int(integrity_rows)):,} ({integrity_pct:.4f}%)")
    print(f"Unique Variants: ~{round_count(integrity_df.count()):,}")
    print("Most common collisions (Valid IDs shared by multiple people):")
    integrity_df.select(kvnr_col, "count").show(20, truncate=False)


# Generate and display report
print_duplicate_report(placeholders_df, integrity_df, total_rows)

BUSINESS ERROR ANALYSIS: DUPLICATE KVNRs
Total Rows Processed: 30,541,347


--- 1. Systematic Placeholders (Invalid & Shared) ---
Affected Rows: 33,617 (0.1101%)


Unique Variants: 2546
Most common placeholders:


+---------------------+-----+-------------------------------------+
|EGKVERSICHERTENNUMMER|count|validation_error                     |
+---------------------+-----+-------------------------------------+
|A123456789           |7716 |Checksum mismatch (Expected 0, Got 9)|
|X000000000           |5179 |NULL                                 |
|X999999999           |4736 |Checksum mismatch (Expected 2, Got 9)|
|A999999999           |2437 |Checksum mismatch (Expected 4, Got 9)|
|A000000000           |2003 |Checksum mismatch (Expected 2, Got 0)|
|M999999999           |1300 |NULL                                 |
|K999999999           |943  |Checksum mismatch (Expected 5, Got 9)|
|Y000000000           |424  |Checksum mismatch (Expected 3, Got 0)|
|A999999994           |377  |NULL                                 |
|K000000000           |258  |Checksum mismatch (Expected 3, Got 0)|
|A123456780           |252  |NULL                                 |
|K990000000           |212  |Checksum mismatch (

Unique Variants: 3820509
Most common collisions (Valid IDs shared by multiple people):


[Stage 130:====================================================>(197 + 3) / 200]

+---------------------+-----+
|EGKVERSICHERTENNUMMER|count|
+---------------------+-----+
|L202056336           |28   |
|W404008837           |25   |
|G240379401           |23   |
|J123456789           |23   |
|C177133822           |21   |
|G599130609           |18   |
|X242502603           |18   |
|J889124870           |18   |
|S513345607           |18   |
|U015467632           |18   |
|M974910791           |18   |
|G465124454           |18   |
|L347346208           |17   |
|V062877226           |17   |
|L884919611           |17   |
|T671531962           |17   |
|I167750467           |17   |
|H866071813           |16   |
|C175945215           |16   |
|V264577135           |16   |
+---------------------+-----+
only showing top 20 rows



## 6. Synthesis Parameters

Compile all extracted statistics into a single exportable structure for synthetic data generation.


In [ ]:
import json

# Compile all synthesis parameters into a single structure
synthesis_params = {
    "metadata": {
        "source_rows_approx": round_count(total_rows),
        "k_anonymity_threshold": K_ANONYMITY,
        "count_rounding": ROUND_TO,
        "privacy_note": "All distributions are k-anonymized (k≥100), binned, and counts rounded"
    },
    
    "valid_data_distributions": {
        "vorname": {
            "frequency_distribution": vorname_dist.to_dict(orient="records"),
            "length_distribution": vorname_lengths.to_dict(orient="records"),
            "character_patterns": {k: v for k, v in vorname_patterns.items() if k != "total"},
            "unique_values_k100": len(vorname_dist)
        },
        "nachname": {
            "frequency_distribution": nachname_dist.to_dict(orient="records"),
            "length_distribution": nachname_lengths.to_dict(orient="records"),
            "character_patterns": {k: v for k, v in nachname_patterns.items() if k != "total"},
            "unique_values_k100": len(nachname_dist)
        },
        "kvnr": {
            "first_letter_distribution": letter_dist.to_dict(orient="records")
        },
        "plz": {
            "regional_distribution": plz_regional.to_dict(orient="records"),
            "stats": {
                "min": int(plz_stats["min_plz"]) if plz_stats["min_plz"] else None,
                "max": int(plz_stats["max_plz"]) if plz_stats["max_plz"] else None,
                "mean": int(plz_stats["mean_plz"]) if plz_stats["mean_plz"] else None,
                "std": int(plz_stats["std_plz"]) if plz_stats["std_plz"] else None
            }
        },
        "geburtsdatum": {
            "birth_year_bins": birth_year_dist.to_dict(orient="records"),
            "month_distribution": month_dist.to_dict(orient="records"),
            "day_distribution": day_dist.to_dict(orient="records"),
            "age_stats": {
                "min": int(age_stats["min_age"]) if age_stats["min_age"] else None,
                "max": int(age_stats["max_age"]) if age_stats["max_age"] else None,
                "mean": float(age_stats["mean_age"]) if age_stats["mean_age"] else None,
                "std": float(age_stats["std_age"]) if age_stats["std_age"] else None
            }
        },
        "geschlecht": {
            "distribution": gender_dist.to_dict(orient="records")
        },
        "vsdm": {
            "result_distribution": vsdm_result_dist.to_dict(orient="records"),
            "error_distribution": vsdm_error_dist.to_dict(orient="records"),
            "valid_rate": vsdm_valid_rate,
            "coverage_rate": vsdm_coverage_rate,
            "total_records": vsdm_total
        }
    },
    
    "cardinality": cardinality_df.to_dict(orient="records"),
    
    "error_injection": error_profile,
    
    "error_correlations": error_correlations,
    
    "duplicate_patterns": {
        "placeholder_count": placeholders_df.count(),
        "placeholder_affected_rows": int(placeholders_df.agg(F.sum("count")).collect()[0][0] or 0),
        "integrity_violation_count": integrity_df.count(),
        "integrity_affected_rows": int(integrity_df.agg(F.sum("count")).collect()[0][0] or 0)
    }
}

# Calculate rates for duplicates
synthesis_params["duplicate_patterns"]["placeholder_rate"] = (
    synthesis_params["duplicate_patterns"]["placeholder_affected_rows"] / total_rows
)
synthesis_params["duplicate_patterns"]["integrity_violation_rate"] = (
    synthesis_params["duplicate_patterns"]["integrity_affected_rows"] / total_rows
)

print("=" * 70)
print("SYNTHESIS PARAMETERS - COMPLETE")
print("=" * 70)
print(f"\n✓ Valid data distributions extracted for {len(synthesis_params['valid_data_distributions'])} fields")
print(f"✓ Error profiles computed for {len(synthesis_params['error_injection']['field_errors'])} fields")
print(f"✓ Error correlations computed")
print(f"✓ Cardinality statistics computed")
print(f"✓ Duplicate patterns analyzed")
print(f"\nPrivacy: k≥{K_ANONYMITY}, counts rounded to nearest {ROUND_TO}")
print("\nThe 'synthesis_params' dict contains all parameters needed for generation.")


In [ ]:
# Export synthesis parameters to JSON file
output_file = "synthesis_params.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(synthesis_params, f, indent=2, ensure_ascii=False, default=str)

print(f"✓ Synthesis parameters exported to: {output_file}")
print(f"  File size: {len(json.dumps(synthesis_params, default=str)):,} bytes")
print("\nThis JSON file contains all parameters needed to generate synthetic data.")


In [ ]:
# Print a quick summary of key parameters
print("=" * 70)
print("QUICK REFERENCE: KEY SYNTHESIS PARAMETERS")
print("=" * 70)

print("\n📊 VALID DATA DISTRIBUTIONS (for realistic value generation):")
print(f"   • First names: {len(vorname_dist)} unique values (k≥{K_ANONYMITY})")
print(f"   • Last names: {len(nachname_dist)} unique values (k≥{K_ANONYMITY})")
print(f"   • KVNR first letters: {len(letter_dist)} patterns")
print(f"   • Postal code regions: {len(plz_regional)} (first 2 digits)")
print(f"   • Birth years: {len(birth_year_dist)} 5-year bins")
print(f"   • Birth months: {len(month_dist)}, Birth days: {len(day_dist)}")
print(f"   • Gender values: {len(gender_dist)} values")
print(f"   • VSDM result values: {len(vsdm_result_dist)} values")
print(f"   • VSDM valid rate: {vsdm_valid_rate*100:.1f}%, coverage rate: {vsdm_coverage_rate*100:.1f}%")

print("\n⚠️ ERROR RATES (for realistic error injection):")
for field, stats in sorted(error_profile["field_errors"].items(), 
                           key=lambda x: x[1]["error_rate"], reverse=True):
    print(f"   • {field}: {stats['error_rate']*100:.2f}%")

print(f"\n🔄 DUPLICATE PATTERNS:")
print(f"   • Systematic placeholders: {synthesis_params['duplicate_patterns']['placeholder_rate']*100:.2f}%")
print(f"   • Integrity violations: {synthesis_params['duplicate_patterns']['integrity_violation_rate']*100:.2f}%")

print("\n" + "=" * 70)
print("Run the synthetic data generator with 'synthesis_params.json' as input.")
print("=" * 70)
